# TB Pipeline — Qualitative Comparisons

Generates the visual figures the paper needs (Figures 6–10).

**Two-step workflow:**
1. **Candidate gallery** — runs both Baseline and MoE pipelines on ~10 images per dataset, saves a multi-panel PNG per image to `/kaggle/working/qualitative/candidates/`. Browse these and pick the IDs you want for the final figures.
2. **Final figures** — set `SELECTED_IDS` and re-run; produces the paper figures `lung_masks_panel.png`, `lesion_comparison.png`, `expert_specialization.png`, `cavity_detection.png`, `failure_cases.png`.

## Datasets to attach
Same as `eval_moe.ipynb`.

In [ ]:
# Cell 1: Clone repo + install
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '-b', 'cleaned-repo',
         'https://github.com/mabdullahi7780/dl-project-codebase.git', str(REPO_DIR)],
        check=True
    )
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Repo ready:', REPO_DIR)

In [ ]:
# Cell 2: Paths + device
from pathlib import Path
import torch, yaml

KAGGLE_INPUT = Path('/kaggle/input')
MONTGOMERY_ROOT  = KAGGLE_INPUT / 'datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT    = KAGGLE_INPUT / 'datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT      = KAGGLE_INPUT / 'datasets/usmanshams/tbx-11/TBX11K'
NIH_ROOT         = KAGGLE_INPUT / 'datasets/organizations/nih-chest-xrays/data'
CHECKPOINTS_ROOT = KAGGLE_INPUT / 'datasets/iahmedhabib/tb-pipeline-checkpoints'
MEDSAM_CKPT      = KAGGLE_INPUT / 'datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('/kaggle/working/qualitative')
CAND_DIR = OUT_DIR / 'candidates'
FIG_DIR  = OUT_DIR / 'figures'
for d in (CAND_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)
print('Device:', device)
print('Output dir:', OUT_DIR)

In [ ]:
# Cell 3: Build configs and load all models
import yaml
from pathlib import Path

def _find(*candidates):
    for c in candidates:
        if Path(c).is_file(): return str(c)
    return None

c1_adapter  = _find(CHECKPOINTS_ROOT/'component1/component1_adapters.safetensors',
                    REPO_DIR/'checkpoints/component1/component1_adapters.safetensors')
c4_decoder  = _find(CHECKPOINTS_ROOT/'component4/component4_mask_decoder.pt',
                    REPO_DIR/'checkpoints/component4/component4_mask_decoder.pt')
c2_routing  = _find(CHECKPOINTS_ROOT/'component2/component2_routing_head.pt',
                    REPO_DIR/'checkpoints/component2/component2_routing_head.pt')
moe_ckpt    = _find(CHECKPOINTS_ROOT/'component_moe/moe_best.pt',
                    REPO_DIR/'checkpoints/component_moe/moe_best.pt')
critic_ckpt = _find(CHECKPOINTS_ROOT/'component_moe/boundary_critic.pt',
                    REPO_DIR/'checkpoints/component_moe/boundary_critic.pt')

with (REPO_DIR/'configs/moe.yaml').open() as f:
    cfg = yaml.safe_load(f) or {}
cfg.setdefault('runtime', {})['device'] = str(device)
cfg.setdefault('component1', {}).update({'backend':'auto','checkpoint_path':str(MEDSAM_CKPT),'adapter_path':c1_adapter})
cfg.setdefault('component2', {}).update({'backend':'auto','routing_head_path':c2_routing})
cfg.setdefault('component4', {}).update({'backend':'auto','checkpoint_path':str(MEDSAM_CKPT),'model_type':'vit_b','decoder_checkpoint_path':c4_decoder})
cfg.setdefault('moe', {}).update({'enabled':True,'checkpoint_path':moe_ckpt})
cfg.setdefault('component7_moe', {})['boundary_critic_checkpoint'] = critic_ckpt

from src.app.infer import build_models, build_moe_models
c1, c2, c4 = build_models(cfg, device)
c1.eval(); c2.eval(); c4.eval()
moe = build_moe_models(cfg, device)
routing_gate, expert_bank, fusion, boundary_critic = moe
routing_gate.eval(); expert_bank.eval(); fusion.eval()
print('All models loaded.')

In [ ]:
# Cell 4: Single-image inference returning all artifacts (CXR + lung + baseline + MoE + per-expert)
import torch, torch.nn.functional as F, numpy as np
from src.components.component0_qc import harmonise_sample
from src.components.component5_experts import EXPERT_NAMES
from src.components.baseline_lesion_proposer import (
    BaselineLesionProposer, BaselineLesionProposerConfig,
)
from src.utils.morphology import adaptive_lesion_threshold

# Build the baseline lesion proposer once. If C2 has a trained TB head,
# its weight vector gives a TB-specific CAM (matches eval_baseline behavior).
_blp_cfg = BaselineLesionProposerConfig(
    suspicious_class_threshold=cfg.get('baseline_lesion_proposer', {}).get('suspicious_class_threshold', 0.55),
    fixed_binary_threshold=cfg.get('baseline_lesion_proposer', {}).get('fixed_binary_threshold'),
    min_region_px=cfg.get('baseline_lesion_proposer', {}).get('min_region_px', 48),
    opening_iters=cfg.get('baseline_lesion_proposer', {}).get('opening_iters', 1),
    closing_iters=cfg.get('baseline_lesion_proposer', {}).get('closing_iters', 1),
    fallback_blend=cfg.get('baseline_lesion_proposer', {}).get('fallback_blend', 0.35),
)
_tb_head_weight = c2.tb_head.weight.detach() if hasattr(c2, 'tb_head') else None
baseline_proposer = BaselineLesionProposer(_blp_cfg, tb_head_weight=_tb_head_weight)

@torch.no_grad()
def run_full(image_np, dataset_id):
    """Run both Baseline and MoE on one image. Returns dict of arrays for plotting."""
    h = harmonise_sample({'image': image_np, 'dataset_id': dataset_id})
    x3   = h.x_3ch.unsqueeze(0).to(device)
    x224 = h.x_224.unsqueeze(0).to(device)
    x1024= h.x_1024.to(device)

    img_emb, _ = c1(x3, lambda_=0.0)
    txv  = c2.forward_features(x224)
    lung = c4.predict_masks(x3)
    lung_256  = lung.lung_mask_256[0,0].detach().cpu().numpy()
    lung_1024 = lung.lung_mask_1024[0,0].detach().cpu().numpy()

    # Baseline lesion (Grad-CAM via TB head weight if available)
    tb_logit = None
    if hasattr(c2, 'tb_head'):
        tb_logit = c2.tb_head(txv.pooled_features)
    baseline_out = baseline_proposer.propose(
        x_224=x224,
        features_7x7=txv.features_7x7,
        pathology_logits=txv.pathology_logits,
        lung_mask_256=lung.lung_mask_256,
        classifier_weight=txv.classifier_weight,
        tb_logit=tb_logit,
    )
    # baseline_out.lesion_mask_coarse_256 shape [1,1,256,256]
    baseline_lesion_256 = baseline_out.lesion_mask_coarse_256[0,0].detach().cpu().numpy()
    baseline_lesion_1024 = np.kron(baseline_lesion_256, np.ones((4,4)))[:1024,:1024]

    # MoE
    rw = routing_gate(img_emb, txv.domain_ctx if routing_gate.use_domain_ctx else None)
    expert_logits = expert_bank(img_emb)  # list of K [1,1,256,256]
    fusion_out = fusion(expert_logits, rw)
    fused_prob_256 = fusion_out.mask_fused_256[0,0].detach().cpu().numpy()
    thr = adaptive_lesion_threshold(fused_prob_256, lung_256)
    moe_lesion_256 = (fused_prob_256 > thr).astype(np.uint8)

    # Per-expert sigmoid maps (for Fig 8)
    expert_probs = {EXPERT_NAMES[k]: torch.sigmoid(expert_logits[k])[0,0].detach().cpu().numpy()
                    for k in range(len(EXPERT_NAMES)) if k < expert_bank.num_experts}
    cavity_prob = expert_probs.get('cavity')
    cavity_bin  = (cavity_prob > 0.65).astype(np.uint8) if cavity_prob is not None else None

    return {
        'cxr_1024'      : x3[0,0].detach().cpu().numpy(),
        'lung_256'      : lung_256, 'lung_1024': lung_1024,
        'baseline_lesion_256': baseline_lesion_256,
        'baseline_lesion_1024': baseline_lesion_1024,
        'moe_lesion_256': moe_lesion_256, 'fused_prob_256': fused_prob_256,
        'expert_probs'  : expert_probs,
        'cavity_prob'   : cavity_prob, 'cavity_bin': cavity_bin,
        'routing_weights': {EXPERT_NAMES[k]: float(rw[0,k].item()) for k in range(rw.shape[1])},
    }

print('Inference helper ready.')

In [ ]:
# Cell 5: Generate the candidate gallery — N images per dataset
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.evaluation.baseline_eval import (
    DOMAIN_IDS, build_eval_manifest, make_test_splits, _load_image_for_entry,
    _load_montgomery_lung_gt_1024, _load_shenzhen_lung_gt_1024, load_tbx11k_bbox_index,
)

N_PER_DATASET = 10  # increase for more candidates

samples = build_eval_manifest(
    montgomery_root=MONTGOMERY_ROOT, shenzhen_root=SHENZHEN_ROOT,
    tbx11k_root=TBX11K_ROOT, nih_root=NIH_ROOT if NIH_ROOT.exists() else None,
    tbx_list_name='all_trainval.txt', nih_cache_path=OUT_DIR/'_nih_cache.json',
)
splits = make_test_splits(
    samples, seed=42, holdout_frac=0.2, limit_per_domain=200,
    cache_path=OUT_DIR/'test_splits.json',
)
tbx_boxes = load_tbx11k_bbox_index(TBX11K_ROOT)

rng = np.random.default_rng(123)
candidate_records = []

for dom in DOMAIN_IDS:
    pool = splits.get(dom, [])
    if not pool: continue
    # For TB-containing datasets prefer TB-positive cases; for NIH take random
    if dom != 'nih_cxr14':
        tb_pos = [e for e in pool if e.tb_label == 1]
        tb_neg = [e for e in pool if e.tb_label == 0]
        chosen = (
            list(rng.choice(tb_pos, min(len(tb_pos), N_PER_DATASET//2), replace=False)) +
            list(rng.choice(tb_neg, min(len(tb_neg), N_PER_DATASET - N_PER_DATASET//2), replace=False))
        )
    else:
        chosen = list(rng.choice(pool, min(len(pool), N_PER_DATASET), replace=False))

    for entry in chosen:
        try:
            image_np, _ = _load_image_for_entry(entry)
            arts = run_full(image_np, dom)
        except Exception as e:
            print(f'  skip {dom}/{entry.image_path}: {e}')
            continue
        basename = Path(entry.image_path or '').stem or 'unknown'
        candidate_id = f'{dom}__{basename}'

        # Plot 5-panel preview
        fig, axes = plt.subplots(1, 5, figsize=(20, 4))
        axes[0].imshow(arts['cxr_1024'], cmap='gray'); axes[0].set_title(f'CXR\n{candidate_id}\ntb={entry.tb_label}')
        axes[1].imshow(arts['cxr_1024'], cmap='gray'); axes[1].imshow(arts['lung_1024'], cmap='Blues', alpha=0.4); axes[1].set_title('Lung mask overlay')
        axes[2].imshow(arts['cxr_1024'], cmap='gray'); axes[2].imshow(arts['baseline_lesion_1024'], cmap='Reds', alpha=0.5); axes[2].set_title('Baseline (Grad-CAM)')
        axes[3].imshow(arts['cxr_1024'], cmap='gray')
        # MoE lesion at 256 — upsample for overlay
        moe_up = np.kron(arts['moe_lesion_256'], np.ones((4,4))).astype(np.uint8)
        axes[3].imshow(moe_up[:1024,:1024], cmap='Greens', alpha=0.5); axes[3].set_title('MoE fused')
        rw_str = ' '.join(f'{k[:3]}={v:.2f}' for k,v in arts['routing_weights'].items())
        axes[4].axis('off'); axes[4].text(0.05, 0.5, f'Routing weights:\n{rw_str}', fontsize=10)
        for ax in axes[:4]: ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        out_png = CAND_DIR / f'{candidate_id}.png'
        plt.savefig(out_png, dpi=80, bbox_inches='tight')
        plt.close(fig)

        candidate_records.append({
            'id': candidate_id, 'dataset': dom, 'tb_label': entry.tb_label,
            'image_path': entry.image_path,
            'has_tbx_bbox': basename in tbx_boxes if dom == 'tbx11k' else False,
            'png': str(out_png),
        })

import pandas as pd
cand_df = pd.DataFrame(candidate_records)
cand_df.to_csv(OUT_DIR/'candidates_index.csv', index=False)
print(f'\nGenerated {len(cand_df)} candidate previews under {CAND_DIR}')
print('Browse them via the Kaggle Output tab, then list your chosen IDs in the next cell.')

## STOP HERE on the first pass

Download `/kaggle/working/qualitative/candidates/` (or browse via the Output tab), pick the IDs you want, and fill in `SELECTED_IDS` below. Then re-run from this cell onward.

In [ ]:
# Cell 6: User selections — fill in candidate IDs from the gallery
# Use the candidate_id format: '<dataset>__<basename>' (no .png)

SELECTED_IDS = {
    # Fig 6 — Lung mask quality (4 cases: 2 Montgomery, 2 Shenzhen — pick clean ones)
    'lung_quality': [
        # 'montgomery__MCUCXR_0001_0',
        # 'montgomery__MCUCXR_0250_1',
        # 'shenzhen__CHNCXR_0001_0',
        # 'shenzhen__CHNCXR_0500_1',
    ],
    # Fig 7 — Lesion comparison (6 cases: pick a mix of TBX11K with bbox GT, Shenzhen TB+, NIH healthy)
    'lesion_compare': [],
    # Fig 8 — Per-expert specialization (3 Shenzhen TB+ cases preferred)
    'expert_spec': [],
    # Fig 9 — Cavity detection (3 cases known to have cavity from Shenzhen)
    'cavity_demo': [],
    # Fig 10 — Failure cases (1 NIH FP, 1 Montgomery noise)
    'failure': [],
}

missing = [k for k, v in SELECTED_IDS.items() if not v]
if missing:
    print(f'No IDs selected for: {missing}. Fill in SELECTED_IDS above and re-run.')
else:
    print('All selections present:', {k: len(v) for k, v in SELECTED_IDS.items()})

In [ ]:
# Cell 7: Helpers to load a selected ID and render final figures
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

cand_df = pd.read_csv(OUT_DIR/'candidates_index.csv')
id_to_entry = {row['id']: row for _, row in cand_df.iterrows()}

def load_artifacts_for_id(cand_id):
    if cand_id not in id_to_entry:
        raise KeyError(f'Unknown candidate id: {cand_id}')
    row = id_to_entry[cand_id]
    # Reconstruct entry-like and rerun inference (cheap, ~1.5s/img)
    image_np, _ = _load_image_for_entry(type('E', (), dict(
        dataset_id=row['dataset'], image_path=row['image_path'],
        member_name=None, tb_label=row['tb_label'], domain_id=None,
    )))
    return run_full(image_np, row['dataset']), row

def _overlay(ax, gray, mask, color, alpha=0.5):
    ax.imshow(gray, cmap='gray')
    if mask is not None:
        ax.imshow(mask, cmap=color, alpha=alpha)
    ax.set_xticks([]); ax.set_yticks([])

print('Helpers ready.')

In [ ]:
# Cell 8: Fig 6 — Lung mask quality panel
ids = SELECTED_IDS['lung_quality']
if not ids:
    print('Skip Fig 6 — no IDs selected')
else:
    fig, axes = plt.subplots(len(ids), 3, figsize=(11, 3.5*len(ids)))
    if len(ids) == 1: axes = axes[None, :]
    for i, cid in enumerate(ids):
        arts, row = load_artifacts_for_id(cid)
        # GT lung mask (only Mont/Shen)
        gt = None
        if row['dataset'] == 'montgomery':
            gt = _load_montgomery_lung_gt_1024(Path(row['image_path']))
        elif row['dataset'] == 'shenzhen':
            gt = _load_shenzhen_lung_gt_1024(Path(row['image_path']))
        axes[i, 0].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 0].set_title(f'{row["dataset"]} — CXR'); axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        if gt is not None:
            _overlay(axes[i, 1], arts['cxr_1024'], gt, 'Blues', 0.4); axes[i, 1].set_title('Ground-truth lung')
        else:
            axes[i, 1].axis('off'); axes[i, 1].text(0.5, 0.5, 'no GT', ha='center')
        _overlay(axes[i, 2], arts['cxr_1024'], arts['lung_1024'], 'Greens', 0.4); axes[i, 2].set_title('Our predicted lung')
    plt.tight_layout()
    out = FIG_DIR / 'lung_masks_panel.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
# Cell 9: Fig 7 — Lesion mask comparison (Baseline vs MoE) — HEADLINE FIGURE
from src.evaluation.baseline_eval import _rasterise_boxes_to_256
import torch.nn.functional as F

ids = SELECTED_IDS['lesion_compare']
if not ids:
    print('Skip Fig 7 — no IDs selected')
else:
    n = len(ids)
    fig, axes = plt.subplots(n, 5, figsize=(18, 3.5*n))
    if n == 1: axes = axes[None, :]
    for i, cid in enumerate(ids):
        arts, row = load_artifacts_for_id(cid)
        # Upsample MoE lesion 256→1024 for overlay
        moe_up = np.kron(arts['moe_lesion_256'], np.ones((4,4)))[:1024,:1024]
        # GT bbox mask (only TBX11K)
        gt_mask = None
        if row['dataset'] == 'tbx11k' and row['has_tbx_bbox']:
            basename = Path(row['image_path']).name
            boxes = tbx_boxes.get(basename, [])
            if boxes:
                # Need original size — get from running C0 again would be wasteful; use 1024 as proxy
                gt_mask = _rasterise_boxes_to_256(boxes, (1024, 1024))
                gt_mask = np.kron(gt_mask, np.ones((4,4)))[:1024,:1024]

        axes[i, 0].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 0].set_title(f'{row["dataset"]}\ntb={row["tb_label"]}'); axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        _overlay(axes[i, 1], arts['cxr_1024'], arts['lung_1024'], 'Blues', 0.35); axes[i, 1].set_title('Lung mask')
        _overlay(axes[i, 2], arts['cxr_1024'], arts['baseline_lesion_1024'], 'Reds', 0.5); axes[i, 2].set_title('Baseline (Grad-CAM)')
        _overlay(axes[i, 3], arts['cxr_1024'], moe_up, 'Greens', 0.5); axes[i, 3].set_title('MoE fused')
        if gt_mask is not None:
            _overlay(axes[i, 4], arts['cxr_1024'], gt_mask, 'YlOrBr', 0.5); axes[i, 4].set_title('TBX11K bbox GT')
        else:
            axes[i, 4].axis('off'); axes[i, 4].text(0.5, 0.5, '(no GT)', ha='center')
    plt.tight_layout()
    out = FIG_DIR / 'lesion_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
# Cell 10: Fig 8 — Per-expert specialisation (4 expert outputs side by side)
ids = SELECTED_IDS['expert_spec']
if not ids:
    print('Skip Fig 8 — no IDs selected')
else:
    n = len(ids)
    fig, axes = plt.subplots(n, 5, figsize=(18, 3.5*n))
    if n == 1: axes = axes[None, :]
    expert_order = ['consolidation', 'cavity', 'fibrosis', 'nodule']
    cmaps = ['Reds', 'Purples', 'Oranges', 'Greens']
    for i, cid in enumerate(ids):
        arts, row = load_artifacts_for_id(cid)
        axes[i, 0].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 0].set_title(f'{row["dataset"]}\n{Path(row["image_path"]).stem}'); axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        for j, exp in enumerate(expert_order):
            prob = arts['expert_probs'].get(exp)
            if prob is None:
                axes[i, j+1].axis('off'); continue
            prob_up = np.kron(prob, np.ones((4,4)))[:1024,:1024]
            axes[i, j+1].imshow(arts['cxr_1024'], cmap='gray')
            axes[i, j+1].imshow(prob_up, cmap=cmaps[j], alpha=0.55, vmin=0, vmax=1)
            w = arts['routing_weights'].get(exp, 0.0)
            axes[i, j+1].set_title(f'{exp.title()}  (w={w:.2f})')
            axes[i, j+1].set_xticks([]); axes[i, j+1].set_yticks([])
    plt.tight_layout()
    out = FIG_DIR / 'expert_specialization.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
# Cell 11: Fig 9 — Cavity detection (Expert 2)
ids = SELECTED_IDS['cavity_demo']
if not ids:
    print('Skip Fig 9 — no IDs selected')
else:
    n = len(ids)
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.5*n))
    if n == 1: axes = axes[None, :]
    for i, cid in enumerate(ids):
        arts, row = load_artifacts_for_id(cid)
        axes[i, 0].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 0].set_title(f'{row["dataset"]}\nTB+'); axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        if arts['cavity_prob'] is not None:
            cav_up = np.kron(arts['cavity_prob'], np.ones((4,4)))[:1024,:1024]
            axes[i, 1].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 1].imshow(cav_up, cmap='hot', alpha=0.6, vmin=0, vmax=1); axes[i, 1].set_title('Expert 2 prob')
            cav_bin_up = np.kron(arts['cavity_bin'], np.ones((4,4)))[:1024,:1024]
            axes[i, 2].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 2].imshow(cav_bin_up, cmap='Reds', alpha=0.5); axes[i, 2].set_title('Cavity binary (τ=0.65)')
        else:
            axes[i, 1].axis('off'); axes[i, 2].axis('off')
        for ax in axes[i]: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    out = FIG_DIR / 'cavity_detection.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
# Cell 12: Fig 10 — Failure cases
ids = SELECTED_IDS['failure']
if not ids:
    print('Skip Fig 10 — no IDs selected')
else:
    n = len(ids)
    fig, axes = plt.subplots(n, 4, figsize=(15, 3.5*n))
    if n == 1: axes = axes[None, :]
    for i, cid in enumerate(ids):
        arts, row = load_artifacts_for_id(cid)
        moe_up = np.kron(arts['moe_lesion_256'], np.ones((4,4)))[:1024,:1024]
        axes[i, 0].imshow(arts['cxr_1024'], cmap='gray'); axes[i, 0].set_title(f'{row["dataset"]}\ntb={row["tb_label"]}'); axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        _overlay(axes[i, 1], arts['cxr_1024'], arts['lung_1024'], 'Blues', 0.35); axes[i, 1].set_title('Lung mask')
        _overlay(axes[i, 2], arts['cxr_1024'], moe_up, 'Greens', 0.5); axes[i, 2].set_title('MoE prediction')
        # Annotation panel — describe failure mode
        rw_str = ', '.join(f'{k[:3]}={v:.2f}' for k,v in arts['routing_weights'].items())
        axes[i, 3].axis('off')
        axes[i, 3].text(0.05, 0.5, f'Failure mode:\n(annotate per case)\n\nRouting:\n{rw_str}', fontsize=10)
    plt.tight_layout()
    out = FIG_DIR / 'failure_cases.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

In [ ]:
# Cell 13: List all output files for download
print('=' * 60)
print('QUALITATIVE OUTPUTS — for download from /kaggle/working/qualitative/')
print('=' * 60)
print('\nCANDIDATE GALLERY (browse to pick):')
for p in sorted(CAND_DIR.glob('*.png')):
    print(f'  candidates/{p.name}')
print('\nFINAL FIGURES (for paper):')
for p in sorted(FIG_DIR.glob('*.png')):
    print(f'  figures/{p.name}')